In [0]:
%pip install --upgrade openai

In [0]:
dbutils.library.restartPython()

In [0]:
import base64
from PIL import Image
from io import BytesIO
from config import DeployConfig
from openai import OpenAI

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
image_table = getattr(cfg, f"image_table")
brand_table = getattr(cfg, f"brand_table")

In [0]:
openai_client = OpenAI(api_key=dbutils.secrets.get("shovakeemian-scope", "openai-key"))

In [0]:
pet_image=spark.sql(f'select content from {image_table.path} where id=25').collect()[0]['content']
Image.open(BytesIO(pet_image))

In [0]:
# Get the size of the image
image_size = Image.open(BytesIO(pet_image)).size

# Display the image size
display(image_size)

In [0]:
len(bytes(pet_image))

In [0]:
brand_image=spark.sql(f'select content from {brand_table.path} where version=1').collect()[0]['content']
Image.open(BytesIO(brand_image))

In [0]:
# https://platform.openai.com/docs/guides/image-generation?image-generation-model=gpt-image-1#create-a-new-image-using-image-references
#around 7 to 32 cents an image

prompt = """
create an advertisement for pet food with the images provided. Make the pet fromt he pet_image peak out the side of a bag of pet food found in the petfood_brand image. Make it as realistic as possible. Keep the pet in the image's original environemnt and try and incorporate the background.
"""

pet_image_bytes=bytes(pet_image)
brand_image_bytes=bytes(brand_image)

result = openai_client.images.edit(
    model="gpt-image-1",
    image=[("pet_image.png", pet_image_bytes, "image/jpeg"), ("petfood_brand.png", brand_image_bytes, "image/png")],
    prompt=prompt
)

In [0]:
type(image_bytes)

In [0]:
image_base64 = result.data[0].b64_json
image_bytes = base64.b64decode(image_base64)
Image.open(BytesIO(image_bytes))